## Construct CORPUS table

First I am going to look at the files that make up each of the StandardEbooks.


For Giant's Bread which has book and chapter separations, I am going to do the following: as book in the giants bread is a product of publishing (i believe) rather than narrative, I will just continue the numbering making 2-1 be the next number after 1-x ends (REWRITE)

After running into issues in the sentiment analysis stage, I have returned here to swap over to spaCy from NLTK to have lemmatization and a better POS tagger for prose fiction.

In [14]:
# import libraries
import pandas as pd
import re
import spacy
from pathlib import Path
from bs4 import BeautifulSoup

nlp = spacy.load('en_core_web_lg', disable=['ner']) # NER is disabled bc I don't use it and its slow

In [15]:
# get list of all files in the raw_data folder
root_dir = 'raw_data'
file_dict = {}

# get paths to xhtml files
paths = sorted(Path(root_dir).glob('*/src/epub/text/*.xhtml')) # glob finds all the pathnames matching a specified pattern

for path in paths:
    book_id = path.parts[1].replace('agatha-christie_', '') # .parts breaks a path into a tuple of components
    # for each book_id, get the file path to all xhtml files
    if book_id not in file_dict:
        file_dict[book_id] = []
    file_dict[book_id].append(path)

# file_dict

### Write helper functions

In [16]:
BOILERPLATE = {'colophon', 'imprint', 'titlepage', 'uncopyright', 
               'halftitlepage', 'loi', 'dedication'}

def is_content(path):
    stem = path.stem
    if stem in BOILERPLATE:
        return False
    if stem.startswith('book-'):
        return False
    return True

In [17]:
GIANTS_BREAD_CHAPTER_COUNTS = {1: 8, 2: 7, 3: 4, 4: 4, 5: 5}

# make helper function
def get_chap_num(book_id, stem):
    if stem == 'prologue':
        chap_num = 0
    elif book_id == 'giants-bread' and stem.startswith('chapter-'):
        # get the numbers
        N = int(stem.split('-')[1])
        M = int(stem.split('-')[2])
        # cumulative offset for giants bread chapters
        chap_num = sum(count for book, count in GIANTS_BREAD_CHAPTER_COUNTS.items() if book < N) + M
    elif stem.startswith('chapter-'):
        # get the number
        N = int(stem.split('-')[1])
        chap_num = N
    else:
        raise ValueError(f"Cannot determine chap_num for book_id='{book_id}', stem='{stem}'")
    
    return chap_num

In [18]:
# test
print(get_chap_num('the-big-four', 'chapter-5'))        # expect 5
print(get_chap_num('giants-bread', 'chapter-2-3'))      # expect 11
print(get_chap_num('the-secret-adversary', 'prologue')) # expect 0

5
11
0


In [19]:
def tokenize_file(book_id, chap_num, xhtml_path):
    """
    Returns a TOKENS dataframe for one file.
    Uses spaCy for tokenization, sentence segmentation, 
    POS tagging (Penn Treebank), and lemmatization.
    """
    # open file and parse with BeautifulSoup
    with open(xhtml_path, "r") as f:
        soup = BeautifulSoup(f, 'xml')
    # find all <p> tags, filter out chapter titles
    text_paras = [p.get_text() for p in soup.find_all('p')
                  if 'title' not in p.get('epub:type', '')]

    records = []
    for para_num, para_text in enumerate(text_paras):
        doc = nlp(para_text)
        for sent_num, sent in enumerate(doc.sents):
            token_num = 0
            for token in sent:
                # normalize: lowercase, strip non-alphanumeric
                term = re.sub(r'[\W_]+', '', token.text.lower())
                if term == '':
                    continue  # skip punctuation and whitespace tokens
                records.append({
                    'book_id':   book_id,
                    'chap_num':  chap_num,
                    'para_num':  para_num,
                    'sent_num':  sent_num,
                    'token_num': token_num,
                    'token_str': token.text,
                    'term_str':  term,
                    'lemma':     token.lemma_.lower(),
                    'pos':       token.tag_,
                    'pos_group': token.tag_[:2] if len(token.tag_) >= 2 else token.tag_,
                })
                token_num += 1

    TOKENS = pd.DataFrame(records).set_index(
        ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
    )
    return TOKENS

In [20]:
# test
tokenize_file('the-big-four', 1, 'raw_data/agatha-christie_the-big-four/src/epub/text/chapter-1.xhtml')

token_str term_str   lemma  \
book_id      chap_num para_num sent_num token_num                              
the-big-four 1        0        0        0                 I        i       i   
                                        1              have     have    have   
                                        2               met      met    meet   
                                        3            people   people  people   
                                        4               who      who     who   
...                                                     ...      ...     ...   
                      100      1        10                I        i       i   
                                        11              can      can     can   
                                        12           manage   manage  manage   
                                        13              the      the     the   
                                        14             rest     rest    rest   

                                                   pos pos_group  
book_id      chap_num para_num sent_num token_num                 
the-big-four 1        0        0        0          PRP        PR  
                                        1          VBP        VB  
                                        2          VBN        VB  
                                        3          NNS        NN  
                                        4           WP        WP  
...                                                ...       ...  
                      100      1        10         PRP        PR  
                                        11          MD        MD  
                                        12          VB        VB  
                                        13          DT        DT  
                                        14          NN        NN  

[2759 rows x 5 columns]

### Main parsing loop

In [21]:
# tokenize source and append to list of corpora

# initialize list of corpora
corpus = []

for book_id, paths in file_dict.items():
    for path in paths:
        if is_content(path): # if not boilerplate
            print(book_id, path.stem) # see which file is being processed
            # if poirot-investigates, I to handle the short stories (stored the way all other books have chapters) as books themselves
            if book_id == 'poirot-investigates': # this is the book_id from the filepath, not the book_id I use in LIB
                story_id = path.stem
                chap_num = 1
                corpus.append(tokenize_file(story_id, chap_num, path))
            # if any other work
            else:
                chap_num = get_chap_num(book_id, path.stem)
                corpus.append(tokenize_file(book_id, chap_num, path))

CORPUS = pd.concat(corpus) # the dataframes already have the full OHCO index set (do not set here or we end up with another level)
print("\nDone!")
# cleanup
del(corpus)

giants-bread chapter-1-1
giants-bread chapter-1-2
giants-bread chapter-1-3
giants-bread chapter-1-4
giants-bread chapter-1-5
giants-bread chapter-1-6
giants-bread chapter-1-7
giants-bread chapter-1-8
giants-bread chapter-2-1
giants-bread chapter-2-2
giants-bread chapter-2-3
giants-bread chapter-2-4
giants-bread chapter-2-5
giants-bread chapter-2-6
giants-bread chapter-2-7
giants-bread chapter-3-1
giants-bread chapter-3-2
giants-bread chapter-3-3
giants-bread chapter-3-4
giants-bread chapter-4-1
giants-bread chapter-4-2
giants-bread chapter-4-3
giants-bread chapter-4-4
giants-bread chapter-5-1
giants-bread chapter-5-2
giants-bread chapter-5-3
giants-bread chapter-5-4
giants-bread chapter-5-5
giants-bread prologue
poirot-investigates the-adventure-of-the-cheap-flat
poirot-investigates the-adventure-of-the-egyptian-tomb
poirot-investigates the-adventure-of-the-italian-nobleman
poirot-investigates the-adventure-of-the-western-star
poirot-investigates the-case-of-the-missing-will
poirot-inv

KeyboardInterrupt: 

In [ ]:
print(CORPUS.index.names)
print(CORPUS.shape)
display(CORPUS.head())

['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
(845575, 5)


token_str term_str   lemma  \
book_id      chap_num para_num sent_num token_num                              
giants-bread 1        0        0        0             There    there   there   
                                        1              were     were      be   
                                        2              only     only    only   
                                        3             three    three   three   
                                        4            people   people  people   

                                                   pos pos_group  
book_id      chap_num para_num sent_num token_num                 
giants-bread 1        0        0        0           EX        EX  
                                        1          VBD        VB  
                                        2           RB        RB  
                                        3           CD        CD  
                                        4          NNS        NN

In [ ]:
# check / test
CORPUS.index.get_level_values('book_id').unique()

Index(['giants-bread', 'the-adventure-of-the-cheap-flat',
       'the-adventure-of-the-egyptian-tomb',
       'the-adventure-of-the-italian-nobleman',
       'the-adventure-of-the-western-star', 'the-case-of-the-missing-will',
       'the-disappearance-of-mr-davenheim',
       'the-jewel-robbery-at-the-grand-metropolitan',
       'the-kidnapped-prime-minister', 'the-million-dollar-bond-robbery',
       'the-mystery-of-hunters-lodge', 'the-tragedy-at-marsdon-manor',
       'the-big-four', 'the-man-in-the-brown-suit',
       'the-murder-at-the-vicarage', 'the-murder-of-roger-ackroyd',
       'the-murder-on-the-links', 'the-mysterious-affair-at-styles',
       'the-mystery-of-the-blue-train', 'the-secret-adversary',
       'the-secret-of-chimneys', 'the-seven-dials-mystery'],
      dtype='str', name='book_id')

In [ ]:
# test a lemma
CORPUS[CORPUS.term_str == 'murdered'][['token_str','lemma','pos']].head()


token_str  \
book_id                           chap_num para_num sent_num token_num             
the-disappearance-of-mr-davenheim 1        77       7        17         murdered   
the-mystery-of-hunters-lodge      1        19       1        12         murdered   
                                           28       0        5          murdered   
the-big-four                      10       14       8        4          murdered   
                                  16       6        2        5          murdered   

                                                                         lemma  \
book_id                           chap_num para_num sent_num token_num           
the-disappearance-of-mr-davenheim 1        77       7        17         murder   
the-mystery-of-hunters-lodge      1        19       1        12         murder   
                                           28       0        5          murder   
the-big-four                      10       14       8        4          murder   
                                  16       6        2        5          murder   

                                                                        pos  
book_id                           chap_num para_num sent_num token_num       
the-disappearance-of-mr-davenheim 1        77       7        17         VBD  
the-mystery-of-hunters-lodge      1        19       1        12         VBN  
                                           28       0        5          VBD  
the-big-four                      10       14       8        4          VBN  
                                  16       6        2        5          VBN

In [ ]:
# test another lemma
CORPUS[CORPUS.term_str == 'frightened'][['token_str','lemma','pos']].head()

token_str       lemma pos
book_id      chap_num para_num sent_num token_num                            
giants-bread 4        39       2        3          frightened  frightened  JJ
                      146      0        4          frightened  frightened  JJ
             7        164      2        5          frightened  frightened  JJ
             9        152      3        6          frightened  frightened  JJ
                      171      0        4          frightened  frightened  JJ

### Save CORPUS to CSV

In [ ]:
CORPUS.to_csv('data/CORPUS.csv', sep='\t', index = True)